# LeetCode 2288 — Apply Discount to Prices bằng Python

Link đề: https://leetcode.com/problems/apply-discount-to-prices/ · Độ khó: **Medium** · Chủ đề: String

Skeleton LeetCode:

```python
class Solution:
    def discountPrices(self, sentence: str, discount: int) -> str:
        ...
```

## 1. Mục tiêu học

1. **Tách từ (tokenize)** một câu chuẩn và xử lý từng từ độc lập rồi ghép lại — mẫu "split → map → join".
2. Viết **bộ nhận dạng** chính xác cho một ngôn ngữ nhỏ: "từ là giá" ⇔ `'$'` + **một hoặc nhiều** chữ số, **không thêm gì khác** (khớp toàn bộ từ, không khớp một phần).
3. Hiểu vì sao bài **không cần số thực**: `price · (100 − discount)` chính là số **xu (cent)** sau giảm giá ⇒ định dạng `"{c // 100}.{c % 100:02d}"` hoàn toàn chính xác.
4. Biết các bẫy định dạng: `round(x, 2)` không cho đủ 2 chữ số thập phân, `f"{x:.2f}"` làm tròn kiểu IEEE-754 (`2.675 → 2.67`), `str.replace` thay nhầm cả giá dài hơn.
5. Dùng **biểu thức chính quy** đúng cách (`fullmatch`, hoặc `re.sub` với ranh giới bằng lookaround) và so sánh với cách tách từ.

## 2. Đề bài & ví dụ

Một **câu** là chuỗi các từ phân tách bởi **đúng một** dấu cách; mỗi từ có thể chứa chữ số, chữ thường và ký tự `'$'`. Một từ **biểu diễn giá** nếu nó là một dãy chữ số đứng sau **một** dấu `'$'` — ví dụ `"$100"`, `"$23"`, `"$6"` là giá; `"100"`, `"$"`, `"$1e5"` thì không.

Cho `sentence` và số nguyên `discount`. Với mỗi từ là giá, giảm `discount%` và cập nhật từ đó trong câu. Mọi giá sau cập nhật phải có **đúng hai chữ số thập phân**. Trả về câu sau khi sửa.

**Ví dụ 1:** `sentence = "there are $1 $2 and 5$ candies in the shop"`, `discount = 50` → `"there are $0.50 $1.00 and 5$ candies in the shop"`.

**Ví dụ 2:** `sentence = "1 2 $3 4 $5 $6 7 8$ $9 $10$"`, `discount = 100` → `"1 2 $0.00 4 $0.00 $0.00 7 8$ $0.00 $10$"`.

**Ràng buộc:** `1 ≤ len(sentence) ≤ 10⁵`; ký tự gồm chữ thường, chữ số, `' '`, `'$'`; không có dấu cách đầu/cuối, các từ cách nhau đúng một dấu cách; giá là số dương **không có số 0 đứng đầu**, tối đa 10 chữ số; `0 ≤ discount ≤ 100`.

In [1]:
import re, random, time
from fractions import Fraction
from decimal import Decimal, ROUND_HALF_UP, ROUND_HALF_EVEN

examples = [("there are $1 $2 and 5$ candies in the shop", 50, "there are $0.50 $1.00 and 5$ candies in the shop"),
            ("1 2 $3 4 $5 $6 7 8$ $9 $10$", 100, "1 2 $0.00 4 $0.00 $0.00 7 8$ $0.00 $10$")]
for s, d, exp in examples:
    print(f"discount={d:>3}: {s!r}\n        -> {exp!r}")

discount= 50: 'there are $1 $2 and 5$ candies in the shop'
        -> 'there are $0.50 $1.00 and 5$ candies in the shop'
discount=100: '1 2 $3 4 $5 $6 7 8$ $9 $10$'
        -> '1 2 $0.00 4 $0.00 $0.00 7 8$ $0.00 $10$'


## 3. Kiến thức nền

### 3.1 `split()` / `split(" ")` / `" ".join(...)`

Câu chuẩn (đúng một dấu cách, không có dấu cách đầu/cuối) ⇒ `" ".join(s.split(" ")) == s`: tách rồi ghép lại **không làm mất thông tin**. Đây là lý do mẫu "split → map → join" an toàn ở bài này.

In [2]:
s = examples[1][0]
print(s.split(" ")); print(" ".join(s.split(" ")) == s)
print("câu không chuẩn:", "a  b".split(" "), "a  b".split(), "->", repr(" ".join("a  b".split())))

['1', '2', '$3', '4', '$5', '$6', '7', '8$', '$9', '$10$']
True
câu không chuẩn: ['a', '', 'b'] ['a', 'b'] -> 'a b'


### 3.2 Nhận dạng "chuỗi toàn chữ số": `isdigit`, `isdecimal`, so sánh ký tự

`"".isdigit()` là **False** — tiện: `"$"` (không có chữ số) bị loại tự nhiên. Lưu ý `isdigit()` chấp nhận cả các ký tự Unicode như `'²'`; bài chỉ có ASCII nên không sao, nhưng kiểm tra `'0' <= c <= '9'` là cách an toàn tuyệt đối.

In [3]:
for t in ["123", "", "1e5", "12$", "²", "٣"]:
    print(f"{t!r:<6} isdigit={t.isdigit()!s:<5} isdecimal={t.isdecimal()!s:<5} ascii={bool(t) and all('0' <= c <= '9' for c in t)}")

'123'  isdigit=True  isdecimal=True  ascii=True
''     isdigit=False isdecimal=False ascii=False
'1e5'  isdigit=False isdecimal=False ascii=False
'12$'  isdigit=False isdecimal=False ascii=False
'²'    isdigit=True  isdecimal=False ascii=False
'٣'    isdigit=True  isdecimal=True  ascii=False


### 3.3 Định dạng hai chữ số thập phân

`f"{x:.2f}"` luôn in **đúng** 2 chữ số sau dấu chấm; `round(x, 2)` trả về số thực, `str` của nó bỏ số 0 cuối. Với số nguyên `c` xu, `f"{c // 100}.{c % 100:02d}"` cho kết quả chính xác mà không đụng tới số thực.

In [4]:
x = 1.0
print(f"{x:.2f}", str(round(x, 2)), str(round(0.5, 2)))
c = 50
print(f"{c // 100}.{c % 100:02d}", f"{7 // 100}.{7 % 100:02d}", f"{123456 // 100}.{123456 % 100:02d}")

1.00 1.0 0.5
0.50 0.07 1234.56


### 3.4 Biểu diễn số thực IEEE-754 và làm tròn

Nhiều số thập phân không biểu diễn chính xác: `2.675` thực ra là `2.67499999…` ⇒ `f"{2.675:.2f}" == "2.67"`. Bài này **không gặp** vấn đề đó (xem mục 6), nhưng đây là lý do nên ưu tiên số nguyên khi có thể.

In [5]:
print(f"{2.675:.2f}", Decimal(2.675), f"{0.125:.2f}", f"{0.375:.2f}")
print(Decimal("2.675").quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))

2.67 2.67499999999999982236431605997495353221893310546875 0.12 0.38
2.68


### 3.5 Biểu thức chính quy: `fullmatch` và lookaround

`re.fullmatch(r"\$\d+", w)` yêu cầu **toàn bộ** từ khớp. Khi làm trên cả câu, cần ranh giới: `(?<!\S)` (trước là đầu chuỗi hoặc khoảng trắng) và `(?!\S)` (sau là cuối chuỗi hoặc khoảng trắng).

In [6]:
for w in ["$12", "$12$", "a$12", "$", "$1e5"]:
    print(f"{w!r:<7} search={bool(re.search(r'[$][0-9]+', w))!s:<5} fullmatch={bool(re.fullmatch(r'[$][0-9]+', w))}")

'$12'   search=True  fullmatch=True
'$12$'  search=True  fullmatch=False
'a$12'  search=True  fullmatch=False
'$'     search=False fullmatch=False
'$1e5'  search=True  fullmatch=False


## 4. Ý tưởng nền tảng / trực giác

Mỗi từ được xử lý **độc lập**: việc một từ có là giá hay không chỉ phụ thuộc vào chính nó. Vậy:

1. Tách câu thành các từ.
2. Với mỗi từ `w`: **nếu** `w[0] == '$'` **và** `w[1:]` khác rỗng **và** toàn chữ số **thì** thay `w` bằng giá mới; **ngược lại** giữ nguyên.
3. Ghép lại bằng một dấu cách.

**Tính giá mới:** giá `p` (đô-la, nguyên), giảm `d%` ⇒ còn `p · (100 − d) / 100` đô-la = **`p · (100 − d)` xu** — một số **nguyên**! Ví dụ `p = 1, d = 50` ⇒ 50 xu ⇒ `"$0.50"`; `p = 2, d = 50` ⇒ 100 xu ⇒ `"$1.00"`.

Bảng phân loại các từ trong ví dụ 2:

| từ | `w[0]=='$'` | `w[1:]` | toàn chữ số? | là giá? | sau giảm 100% |
|---|---|---|---|---|---|
| `1` | ✘ | — | — | ✘ | `1` |
| `$3` | ✔ | `3` | ✔ | ✔ | `$0.00` |
| `8$` | ✘ | — | — | ✘ | `8$` |
| `$10$` | ✔ | `10$` | ✘ | ✘ | `$10$` |

In [7]:
def classify(w):
    return w[:1] == "$" and len(w) > 1 and all("0" <= c <= "9" for c in w[1:])

for w in examples[1][0].split(" ") + ["$", "$1e5", "$$1", "$0", "a$1"]:
    print(f"{w!r:<7} -> {'GIÁ' if classify(w) else 'không'}")

'1'     -> không
'2'     -> không
'$3'    -> GIÁ
'4'     -> không
'$5'    -> GIÁ
'$6'    -> GIÁ
'7'     -> không
'8$'    -> không
'$9'    -> GIÁ
'$10$'  -> không
'$'     -> không
'$1e5'  -> không
'$$1'   -> không
'$0'    -> GIÁ
'a$1'   -> không


Ghi chú: `"$0"` khớp mẫu nhận dạng, nhưng đề đảm bảo giá là **số dương không có 0 đứng đầu**, nên `"$0"` không xuất hiện trong test; nếu xuất hiện thì cách nhận dạng "chỉ cần toàn chữ số" vẫn xử lý được (thành `"$0.00"`).

## 5. Cách 1 — Brute Force / baseline

### 5.1 Tham chiếu "chuẩn mực" bằng `Fraction` (số hữu tỉ chính xác)

Không có vét cạn tổ hợp ở bài chuỗi này; "baseline" là bản **chậm nhưng chắc chắn đúng**: tính giá mới bằng số hữu tỉ, làm tròn nửa lên một cách tường minh, kiểm tra từng ký tự. Dùng làm bộ đối chiếu cho mọi cách khác.

In [8]:
def reference(sentence, discount):
    out = []
    for w in sentence.split(" "):
        if len(w) >= 2 and w[0] == "$" and all(c in "0123456789" for c in w[1:]):
            v = Fraction(int(w[1:])) * Fraction(100 - discount, 100)        # đô-la, chính xác
            cents = v * 100
            q = cents.numerator // cents.denominator
            if (cents - q) * 2 >= 1: q += 1                                  # làm tròn nửa lên (ở đây luôn nguyên)
            w = "$" + str(q // 100) + "." + str(q % 100).rjust(2, "0")
        out.append(w)
    return " ".join(out)

for s, d, exp in examples:
    got = reference(s, d); print(got == exp, repr(got))

True 'there are $0.50 $1.00 and 5$ candies in the shop'
True '1 2 $0.00 4 $0.00 $0.00 7 8$ $0.00 $10$'


**Độ phức tạp:** O(L) phép duyệt ký tự, nhưng hằng số lớn vì `Fraction` (chuẩn hóa ước chung lớn nhất mỗi phép tính). Vẫn đủ nhanh với `L = 10⁵`, nhưng rườm rà.

### 5.2 Baseline 2 — thay thế bằng `str.replace` trên cả câu (SAI, chỉ để thấy vì sao phải làm theo từng từ)

In [9]:
def naive_replace(sentence, discount):
    for w in set(sentence.split(" ")):
        if classify(w):
            sentence = sentence.replace(w, f"${int(w[1:]) * (100 - discount) / 100:.2f}")
    return sentence

s = "$1 $10 $100"
print(naive_replace(s, 50), "| đúng:", reference(s, 50))

$0.50 $0.500 $50.00 | đúng: $0.50 $5.00 $50.00


`replace("$1", …)` thay cả **tiền tố** của `"$10"` và `"$100"` (và kết quả còn phụ thuộc thứ tự duyệt `set`). **Nút thắt:** phép thay phải tôn trọng **ranh giới từ** ⇒ xử lý theo từng từ đã tách, hoặc regex có ranh giới.

## 6. Ý tưởng thuật toán / bước đột phá

**Tách từ + nhận dạng + số nguyên xu.**

- Nhận dạng: `w[0] == '$'` và `w[1:].isdigit()` (chuỗi rỗng ⇒ False ⇒ `"$"` tự loại).
- Giá mới (xu): `c = int(w[1:]) * (100 − discount)` — **không có làm tròn** vì là số nguyên.
- Định dạng: `"$" + str(c // 100) + "." + f"{c % 100:02d}"`.

**Vì sao bản float của doocs (`f"{p * (1 - d/100):.2f}"`) cũng đúng?** Giá trị thật có đúng 2 chữ số thập phân, tức là bội của `0.01`. `p ≤ 10¹⁰`, nên sai số tương đối của phép nhân float (~`10⁻¹⁶`) cho sai số tuyệt đối cỡ `10⁻⁶` — nhỏ hơn rất nhiều so với `0.005` cần để `.2f` làm tròn sang số khác. Nhưng lập luận này **phụ thuộc vào giới hạn 10 chữ số**; với giá 17–18 chữ số, float bắt đầu sai (mục 15.2). Bản số nguyên đúng với mọi độ dài.

**Độ phức tạp:** O(L) — mỗi ký tự được đọc một số hằng lần (tách, kiểm tra, chuyển đổi số tối đa 10 chữ số, ghép).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tách từ + float (cách doocs)

In [10]:
def split_float(sentence, discount):
    ans = []
    for w in sentence.split():
        if w[0] == "$" and w[1:].isdigit():
            w = f"${int(w[1:]) * (1 - discount / 100):.2f}"
        ans.append(w)
    return " ".join(ans)

for s, d, exp in examples:
    print(split_float(s, d) == exp)

True
True


### 7.2 Cách 3 — Tách từ + số nguyên xu (khuyến nghị)

In [11]:
def split_cents(sentence, discount):
    keep = 100 - discount
    ans = []
    for w in sentence.split(" "):
        if len(w) > 1 and w[0] == "$" and w[1:].isdigit():
            c = int(w[1:]) * keep
            w = f"${c // 100}.{c % 100:02d}"
        ans.append(w)
    return " ".join(ans)

for s, d, exp in examples:
    print(split_cents(s, d) == exp)

True
True


`divmod(c, 100)` cho cả thương và dư trong một lần gọi; kết quả giống hệt.

### 7.3 Cách 4 — `re.sub` với ranh giới từ (không cần tách)

In [12]:
PRICE = re.compile(r"(?<!\S)\$([0-9]+)(?!\S)")

def regex_sub(sentence, discount):
    keep = 100 - discount
    def repl(mo):
        c = int(mo.group(1)) * keep
        return f"${c // 100}.{c % 100:02d}"
    return PRICE.sub(repl, sentence)

for s, d, exp in examples:
    print(regex_sub(s, d) == exp)
print(regex_sub("$1 $10 $100 $10$ a$5", 50))

True
True
$0.50 $5.00 $50.00 $10$ a$5


`(?<!\S)` = "ký tự đứng trước **không phải** ký tự khác khoảng trắng" ⇔ trước đó là đầu chuỗi hoặc dấu cách. Không dùng `\b` được: giữa `' '` và `'$'` **không có** ranh giới từ theo nghĩa `\b` (cả hai đều không phải ký tự "word"), nên `r"\b\$\d+\b"` sẽ trượt.

In [13]:
print("dùng \\b:", re.findall(r"\b\$\d+\b", "there are $1 $2"), "| dùng lookaround:", PRICE.findall("there are $1 $2"))

dùng \b: [] | dùng lookaround: ['1', '2']


### 7.4 Cách 5 — Quét một lượt bằng tay (máy trạng thái), không tạo danh sách từ trung gian

Duyệt ký tự, xác định đoạn `[i, j)` của mỗi từ, kiểm tra và ghi ra. Đây là cách sẽ viết ở ngôn ngữ không có `split` tiện lợi, hoặc khi muốn xử lý dòng dữ liệu (stream).

In [14]:
def manual_scan(sentence, discount):
    keep = 100 - discount
    out = []; n = len(sentence); i = 0
    while i < n:
        j = i
        while j < n and sentence[j] != " ": j += 1          # từ là sentence[i:j]
        is_price = sentence[i] == "$" and j - i >= 2
        k = i + 1
        while is_price and k < j:
            if not ("0" <= sentence[k] <= "9"): is_price = False
            k += 1
        if is_price:
            c = int(sentence[i + 1:j]) * keep
            out.append(f"${c // 100}.{c % 100:02d}")
        else:
            out.append(sentence[i:j])
        if j < n: out.append(" ")
        i = j + 1
    return "".join(out)

for s, d, exp in examples:
    print(manual_scan(s, d) == exp)

True
True


## 8. TRACE từng bước

### 8.1 Trace cách 3 trên ví dụ 1

In [15]:
def trace(sentence, discount):
    keep = 100 - discount; ans = []
    for w in sentence.split(" "):
        if len(w) > 1 and w[0] == "$" and w[1:].isdigit():
            p = int(w[1:]); c = p * keep
            new = f"${c // 100}.{c % 100:02d}"
            print(f"  {w!r:<9} GIÁ  p={p:<4} xu = {p}·{keep} = {c:<5} -> {new!r}")
            w = new
        else:
            why = "không bắt đầu bằng $" if not w.startswith("$") else "phần sau $ không toàn chữ số/rỗng"
            print(f"  {w!r:<9} giữ  ({why})")
        ans.append(w)
    return " ".join(ans)

print(repr(trace(*examples[0][:2])))

  'there'   giữ  (không bắt đầu bằng $)
  'are'     giữ  (không bắt đầu bằng $)
  '$1'      GIÁ  p=1    xu = 1·50 = 50    -> '$0.50'
  '$2'      GIÁ  p=2    xu = 2·50 = 100   -> '$1.00'
  'and'     giữ  (không bắt đầu bằng $)
  '5$'      giữ  (không bắt đầu bằng $)
  'candies' giữ  (không bắt đầu bằng $)
  'in'      giữ  (không bắt đầu bằng $)
  'the'     giữ  (không bắt đầu bằng $)
  'shop'    giữ  (không bắt đầu bằng $)
'there are $0.50 $1.00 and 5$ candies in the shop'


### 8.2 Trace với mức giảm lẻ và giá lớn

`discount = 37`, giá `$9999999999` (10 chữ số) ⇒ xu = `9999999999 · 63 = 629999999937` ⇒ `"$6299999999.37"`.

In [16]:
print(repr(trace("buy $9999999999 or $7 or $ or $12a", 37)))
print("float doocs:", split_float("buy $9999999999 or $7 or $ or $12a", 37))

  'buy'     giữ  (không bắt đầu bằng $)
  '$9999999999' GIÁ  p=9999999999 xu = 9999999999·63 = 629999999937 -> '$6299999999.37'
  'or'      giữ  (không bắt đầu bằng $)
  '$7'      GIÁ  p=7    xu = 7·63 = 441   -> '$4.41'
  'or'      giữ  (không bắt đầu bằng $)
  '$'       giữ  (phần sau $ không toàn chữ số/rỗng)
  'or'      giữ  (không bắt đầu bằng $)
  '$12a'    giữ  (phần sau $ không toàn chữ số/rỗng)
'buy $6299999999.37 or $4.41 or $ or $12a'
float doocs: buy $6299999999.37 or $4.41 or $ or $12a


Với `$7` và giảm 37%: `7 · 63 = 441` xu ⇒ `"$4.41"`. Kiểm: `7 · 0.63 = 4.41` ✔.

## 9. Code nộp LeetCode

In [17]:
class Solution:
    def discountPrices(self, sentence: str, discount: int) -> str:
        keep = 100 - discount                       # phần trăm còn lại
        words = sentence.split(" ")
        for i, w in enumerate(words):
            if len(w) > 1 and w[0] == "$" and w[1:].isdigit():
                cents = int(w[1:]) * keep           # giá mới tính bằng xu: số nguyên chính xác
                words[i] = f"${cents // 100}.{cents % 100:02d}"
        return " ".join(words)

In [18]:
sol = Solution()
for s, d, exp in examples:
    got = sol.discountPrices(s, d)
    print(repr(got), "| OK" if got == exp else "| SAI")

'there are $0.50 $1.00 and 5$ candies in the shop' | OK
'1 2 $0.00 4 $0.00 $0.00 7 8$ $0.00 $10$' | OK


## 10. Vì sao thuật toán đúng

**(a) Tách–ghép bảo toàn câu.** Câu chuẩn có dạng `w₁ + " " + w₂ + … + " " + w_k` với mọi `wᵢ` khác rỗng và không chứa dấu cách. `split(" ")` trả về đúng `[w₁, …, w_k]` và `" ".join` dựng lại đúng câu. Vì ta chỉ thay các phần tử là giá, mọi từ khác và mọi dấu cách được giữ nguyên.

**(b) Nhận dạng đúng.** Từ là giá ⇔ (định nghĩa) có dạng `'$' + D` với `D` là dãy **khác rỗng** các chữ số. Điều kiện `len(w) > 1 and w[0] == '$' and w[1:].isdigit()`: `w[0] == '$'` ⇔ ký tự đầu là `$`; `w[1:]` khác rỗng ⇔ `len(w) > 1`; `isdigit()` với đầu vào ASCII ⇔ mọi ký tự thuộc `'0'..'9'`. Một từ như `"$10$"` hay `"$$1"` có ký tự không phải chữ số trong `w[1:]` ⇒ bị loại đúng.

**(c) Giá trị đúng.** Giá `p` giảm `d%` còn `p(100 − d)/100` đô-la. Nhân với 100 ⇒ `c = p(100 − d)` xu, là số nguyên không âm. Biểu diễn thập phân 2 chữ số của `c/100` là `⌊c/100⌋ . (c mod 100)` với phần lẻ đệm 0 thành 2 chữ số — đúng điều `f"{c // 100}.{c % 100:02d}"` in ra. Không có bước làm tròn nào nên không có sai số.

**(d) Không có leading zero thừa:** phần nguyên `c // 100` in bằng `str` của số nguyên ⇒ `0` khi `c < 100` (ví dụ `"$0.50"`), không bao giờ có `"00"`. ∎

## 11. Phân tích Time / Space Complexity

`L = len(sentence) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Tham chiếu `Fraction` | O(L) (hằng số lớn) | O(L) |
| 2. Tách + float | O(L) | O(L) |
| 3. Tách + số nguyên | O(L) | O(L) |
| 4. `re.sub` | O(L) | O(L) |
| 5. Quét tay | O(L) | O(L) |

Mỗi giá tối đa 10 chữ số ⇒ `int()` và phép nhân là O(1) thực tế.

In [19]:
rng = random.Random(2288)
def rand_word(rng):
    t = rng.random()
    if t < 0.4: return "$" + str(rng.randint(1, 10**10 - 1))
    alphabet = "abc019$"
    return "".join(rng.choice(alphabet) for _ in range(rng.randint(1, 8)))

words = []; length = 0
while length < 10**5 - 20:
    w = rand_word(rng); words.append(w); length += len(w) + 1
big = " ".join(words)[:10**5].rstrip(" ")
if big.endswith("$") and False: pass
res = []
for name, f in [("reference", reference), ("split_float", split_float), ("split_cents", split_cents),
                ("regex_sub", regex_sub), ("manual_scan", manual_scan), ("Solution", Solution().discountPrices)]:
    t0 = time.perf_counter(); res.append(f(big, 37)); dt = time.perf_counter() - t0
    print(f"{name:<12} {dt*1000:7.1f} ms")
print("tất cả giống nhau:", len(set(res)) == 1, "| độ dài câu:", len(big))

reference       31.4 ms
split_float      3.0 ms
split_cents      3.3 ms
regex_sub        3.5 ms
manual_scan     10.7 ms
Solution         3.2 ms
tất cả giống nhau: True | độ dài câu: 99983


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, kiểm bằng tham chiếu `Fraction`)

In [20]:
ALL = {"reference": reference, "split_float": split_float, "split_cents": split_cents,
       "regex_sub": regex_sub, "manual_scan": manual_scan, "Solution": Solution().discountPrices}
edge = [(s, d, e) for s, d, e in examples] + [
    ("$", 50, "$"),                                   # chỉ có dấu $
    ("$1", 0, "$1.00"),                               # không giảm
    ("$1", 100, "$0.00"),                             # giảm hết
    ("$1", 99, "$0.01"),                              # một xu
    ("$1", 1, "$0.99"),
    ("$9999999999", 0, "$9999999999.00"),             # giá lớn nhất
    ("$9999999999", 1, "$9899999999.01"),
    ("$$1 $1$ 1$1 $1e5 $01", 50, "$$1 $1$ 1$1 $1e5 $0.50"),   # chỉ "$01" khớp mẫu (đề đảm bảo không xảy ra)
    ("a b c", 30, "a b c"),                           # không có giá
    ("$3 $3 $3", 33, "$2.01 $2.01 $2.01"),            # giá lặp lại
]
for s, d, exp in edge:
    for name, f in ALL.items():
        got = f(s, d)
        assert got == exp, (name, s, d, got, exp)
    print(f"{s!r:<26} d={d:<3} -> {exp!r}")
print("edge OK")

'there are $1 $2 and 5$ candies in the shop' d=50  -> 'there are $0.50 $1.00 and 5$ candies in the shop'
'1 2 $3 4 $5 $6 7 8$ $9 $10$' d=100 -> '1 2 $0.00 4 $0.00 $0.00 7 8$ $0.00 $10$'
'$'                        d=50  -> '$'
'$1'                       d=0   -> '$1.00'
'$1'                       d=100 -> '$0.00'
'$1'                       d=99  -> '$0.01'
'$1'                       d=1   -> '$0.99'
'$9999999999'              d=0   -> '$9999999999.00'
'$9999999999'              d=1   -> '$9899999999.01'
'$$1 $1$ 1$1 $1e5 $01'     d=50  -> '$$1 $1$ 1$1 $1e5 $0.50'
'a b c'                    d=30  -> 'a b c'
'$3 $3 $3'                 d=33  -> '$2.01 $2.01 $2.01'
edge OK


### 12.2 Đối chiếu ngẫu nhiên với tham chiếu (seed cố định)

Bảng chữ cái nhỏ `"$019a"` để thường xuyên sinh ra các từ "gần giống giá" (`"$1$"`, `"$$0"`, `"9$"`…).

In [21]:
rng = random.Random(42)
for _ in range(3000):
    ws = []
    for _ in range(rng.randint(1, 8)):
        if rng.random() < 0.3:
            ws.append("$" + str(rng.randint(1, 10**rng.randint(1, 10) - 1)))
        else:
            ws.append("".join(rng.choice("$019a") for _ in range(rng.randint(1, 5))))
    s = " ".join(ws); d = rng.randint(0, 100)
    ref = reference(s, d)
    for name, f in ALL.items():
        assert f(s, d) == ref, (name, s, d, f(s, d), ref)
print("3000 câu ngẫu nhiên OK")

3000 câu ngẫu nhiên OK


### 12.3 Tính chất: số từ không đổi; từ không phải giá giữ nguyên; giá mới ≤ giá cũ; `discount = 0` chỉ thêm `".00"`

In [22]:
rng = random.Random(7)
for _ in range(1000):
    ws = [("$" + str(rng.randint(1, 10**6))) if rng.random() < 0.5 else "".join(rng.choice("$1a") for _ in range(rng.randint(1, 4)))
          for _ in range(rng.randint(1, 6))]
    s = " ".join(ws); d = rng.randint(0, 100)
    out = Solution().discountPrices(s, d).split(" ")
    assert len(out) == len(ws)
    for a, b in zip(ws, out):
        if classify(a):
            assert Fraction(b[1:]) <= int(a[1:]) and re.fullmatch(r"\$[0-9]+\.[0-9]{2}", b)
        else:
            assert a == b
    assert Solution().discountPrices(s, 0) == " ".join(w + ".00" if classify(w) else w for w in ws)
print("tính chất OK")

tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `round(x, 2)` rồi `str` (thiếu số 0 cuối)

In [23]:
def wrong_round(sentence, discount):
    out = []
    for w in sentence.split(" "):
        if len(w) > 1 and w[0] == "$" and w[1:].isdigit():
            w = "$" + str(round(int(w[1:]) * (100 - discount) / 100, 2))      # SAI: "$1.0", "$0.5"
        out.append(w)
    return " ".join(out)

s, d, _ = examples[0]
print("bản sai :", repr(wrong_round(s, d)), "\nbản đúng:", repr(Solution().discountPrices(s, d)))

bản sai : 'there are $0.5 $1.0 and 5$ candies in the shop' 
bản đúng: 'there are $0.50 $1.00 and 5$ candies in the shop'


**Vì sao sai:** đề yêu cầu **đúng hai** chữ số thập phân. **Sửa:** `f"{x:.2f}"` hoặc định dạng từ số nguyên xu.

### Lỗi 2 — Tính **số tiền được giảm** thay vì giá còn lại

In [24]:
def wrong_amount(sentence, discount):
    out = []
    for w in sentence.split(" "):
        if len(w) > 1 and w[0] == "$" and w[1:].isdigit():
            c = int(w[1:]) * discount                                       # SAI: phải là (100 - discount)
            w = f"${c // 100}.{c % 100:02d}"
        out.append(w)
    return " ".join(out)

print("bản sai :", repr(wrong_amount("$10 $3", 30)), "| bản đúng:", repr(Solution().discountPrices("$10 $3", 30)))
print("ví dụ 1 (trùng hợp vì 50% = 100% - 50%):", wrong_amount(*examples[0][:2]) == examples[0][2])

bản sai : '$3.00 $0.90' | bản đúng: '$7.00 $2.10'
ví dụ 1 (trùng hợp vì 50% = 100% - 50%): True


**Vì sao sai:** với `discount = 50` kết quả **trùng hợp** đúng (50% giảm = 50% còn lại) nên ví dụ 1 không phát hiện được lỗi; ví dụ 2 thì phát hiện (100% giảm ⇒ còn 0, bản sai cho giá gốc). **Sửa:** nhân với `100 − discount`.

### Lỗi 3 — Nhận dạng giá bằng `search` / `startswith` thay vì khớp **toàn bộ** từ

In [25]:
def wrong_search(sentence, discount):
    out = []
    for w in sentence.split(" "):
        mo = re.search(r"\$([0-9]+)", w)                                   # SAI: khớp một phần
        if mo:
            c = int(mo.group(1)) * (100 - discount)
            w = f"${c // 100}.{c % 100:02d}"
        out.append(w)
    return " ".join(out)

s, d, _ = examples[1]
print("bản sai :", repr(wrong_search(s, d)), "\nbản đúng:", repr(Solution().discountPrices(s, d)))

bản sai : '1 2 $0.00 4 $0.00 $0.00 7 8$ $0.00 $0.00' 
bản đúng: '1 2 $0.00 4 $0.00 $0.00 7 8$ $0.00 $10$'


**Vì sao sai:** `"$10$"` chứa đoạn `"$10"` nên bị coi là giá. **Sửa:** `re.fullmatch` hoặc kiểm tra `w[1:].isdigit()`.

### Lỗi 4 — `str.replace` trên cả câu (thay nhầm tiền tố của giá khác)

In [26]:
s = "$1 $10 $100"
print("bản sai :", repr(naive_replace(s, 50)), "| bản đúng:", repr(Solution().discountPrices(s, 50)))

bản sai : '$0.50 $0.500 $50.00' | bản đúng: '$0.50 $5.00 $50.00'


**Sửa:** xử lý từng từ sau khi tách, hoặc regex có ranh giới `(?<!\S)…(?!\S)`.

### Lỗi 5 — Dùng `\b` làm ranh giới trong regex

In [27]:
def wrong_word_boundary(sentence, discount):
    keep = 100 - discount
    return re.sub(r"\b\$([0-9]+)\b", lambda mo: f"${int(mo.group(1)) * keep // 100}.{int(mo.group(1)) * keep % 100:02d}", sentence)

s, d, _ = examples[0]
print("bản sai :", repr(wrong_word_boundary(s, d)), "\nbản đúng:", repr(Solution().discountPrices(s, d)))
print("bản sai :", repr(wrong_word_boundary("5$1 $2", 50)), "| bản đúng:", repr(Solution().discountPrices("5$1 $2", 50)))

bản sai : 'there are $1 $2 and 5$ candies in the shop' 
bản đúng: 'there are $0.50 $1.00 and 5$ candies in the shop'
bản sai : '5$0.50 $2' | bản đúng: '5$1 $1.00'


**Vì sao sai:** `\b` là ranh giới giữa ký tự "word" (`\w`) và "non-word". Giữa dấu cách và `$` (đều là non-word) **không có** `\b` ⇒ `"$1"` đứng sau dấu cách không khớp; ngược lại `"5$1"` có `\b` giữa `5` và `$` ⇒ khớp nhầm. **Sửa:** lookaround dựa trên khoảng trắng.

### Lỗi 6 — Quên trường hợp từ chỉ là `"$"` (truy cập `w[1]` hoặc `int("")`)

In [28]:
def wrong_lone_dollar(sentence, discount):
    out = []
    for w in sentence.split(" "):
        if w[0] == "$" and all(c.isdigit() for c in w[1:]):                # SAI: all([]) == True với w == "$"
            c = int(w[1:]) * (100 - discount)
            w = f"${c // 100}.{c % 100:02d}"
        out.append(w)
    return " ".join(out)

try:
    print("bản sai :", wrong_lone_dollar("pay $ now", 10))
except Exception as e:
    print("bản sai : lỗi", type(e).__name__, "-", e)
print("bản đúng:", repr(Solution().discountPrices("pay $ now", 10)))

bản sai : lỗi ValueError - invalid literal for int() with base 10: ''
bản đúng: 'pay $ now'


**Vì sao sai:** `all(...)` trên dãy rỗng là `True`, nên `"$"` lọt qua và `int("")` ném `ValueError`. **Sửa:** thêm `len(w) > 1` (hoặc dùng `w[1:].isdigit()`, vốn trả `False` cho chuỗi rỗng).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài "sửa một số từ trong câu" ⇒ **split → map → join**; chỉ dùng regex khi cần, và luôn khớp **toàn bộ** từ.
- Tiền / phần trăm với kết quả 2 chữ số thập phân ⇒ nghĩ ngay tới **số nguyên xu**: `p · (100 − d)`.
- `f"{x:.2f}"` đảm bảo đủ 2 chữ số; `round` thì không.
- `"".isdigit()` là `False`, nhưng `all(... for c in "")` là `True` — cẩn thận dãy rỗng.
- Regex: `\b` không dùng được quanh ký tự không phải chữ/số như `$`; dùng `(?<!\S)` và `(?!\S)`.
- Bài cùng họ: 1108 Defanging an IP Address, 824 Goat Latin, 1816 Truncate Sentence, 2129 Capitalize the Title, 68 Text Justification, 8 String to Integer (atoi) — nhận dạng/biến đổi token.

## 15. Biến thể của bài

### 15.1 Biến thể A — Giá có phần xu (`"$12.34"`), mức giảm nguyên, làm tròn **nửa lên**

Giá có xu ⇒ `p · (100 − d)` là số **phần vạn đô-la** (2 + 2 chữ số thập phân) ⇒ phải làm tròn về 2 chữ số. Dùng số nguyên: `cents = (x + 50) // 100` với `x = giá_xu · (100 − d)`.

In [29]:
PRICE_CENTS = re.compile(r"\$(\d+)(?:\.(\d{2}))?")

def discount_with_cents(sentence, discount, rounding="half_up"):
    keep = 100 - discount; out = []
    for w in sentence.split(" "):
        mo = PRICE_CENTS.fullmatch(w)
        if mo:
            pc = int(mo.group(1)) * 100 + int(mo.group(2) or 0)              # giá tính bằng xu
            x = pc * keep                                                   # đơn vị: 1/100 xu
            q, r = divmod(x, 100)
            if rounding == "half_up" and r >= 50: q += 1
            if rounding == "half_even" and (r > 50 or (r == 50 and q % 2 == 1)): q += 1
            w = f"${q // 100}.{q % 100:02d}"
        out.append(w)
    return " ".join(out)

def discount_with_cents_decimal(sentence, discount, mode):
    out = []
    for w in sentence.split(" "):
        if PRICE_CENTS.fullmatch(w):
            v = (Decimal(w[1:]) * (100 - discount) / 100).quantize(Decimal("0.01"), rounding=mode)
            w = f"${v}"
        out.append(w)
    return " ".join(out)

rng = random.Random(151)
for _ in range(2000):
    s = " ".join(f"${rng.randint(0, 999)}.{rng.randint(0, 99):02d}" if rng.random() < 0.7 else "x" for _ in range(4))
    d = rng.randint(0, 100)
    assert discount_with_cents(s, d) == discount_with_cents_decimal(s, d, ROUND_HALF_UP)
    assert discount_with_cents(s, d, "half_even") == discount_with_cents_decimal(s, d, ROUND_HALF_EVEN)
print(discount_with_cents("$0.05 $0.15 $2.50", 50), "|", discount_with_cents("$0.05 $0.15 $2.50", 50, "half_even"))
print("float .2f :", [f"{v * 0.5:.2f}" for v in (0.05, 0.15, 2.50)])

$0.03 $0.08 $1.25 | $0.02 $0.08 $1.25
float .2f : ['0.03', '0.07', '1.25']


`$0.05` giảm 50% = `0.025`: nửa lên ⇒ `0.03`, nửa chẵn (banker's) ⇒ `0.02`. `$0.15` giảm 50% = `0.075`: cả hai quy tắc cho `0.08`, nhưng float `.2f` in **`0.07`** vì `0.15 · 0.5` trong nhị phân là `0.07499999…`. Đây là lý do tiền tệ thực tế dùng số nguyên hoặc `Decimal`, và khai báo rõ quy tắc làm tròn.

### 15.2 Biến thể B — Giá rất dài (tới 20 chữ số): float sai, số nguyên vẫn đúng

In [30]:
long_cases = []
rng = random.Random(152)
for _ in range(3000):
    p = rng.randint(10**15, 10**20); d = rng.randint(1, 99)
    s = f"${p}"
    if split_float(s, d) != split_cents(s, d):
        long_cases.append((s, d))
print("số ca float sai / 3000:", len(long_cases))
s, d = long_cases[0]
print("ví dụ:", s, "d =", d)
print("  float :", split_float(s, d)); print("  nguyên:", split_cents(s, d)); print("  Fraction:", reference(s, d))

số ca float sai / 3000: 2998
ví dụ: $69292383916635585767 d = 81
  float : $13165552944160757760.00
  nguyên: $13165552944160761295.73
  Fraction: $13165552944160761295.73


### 15.3 Biến thể C — Mức giảm khác nhau theo ngưỡng giá (ví dụ: giá ≥ 100 giảm `d₁%`, còn lại `d₂%`)

Chỉ hàm "tính giá mới" thay đổi; khung split → map → join giữ nguyên — tách riêng **nhận dạng** và **biến đổi** giúp mở rộng dễ.

In [31]:
def tiered(sentence, d_high, d_low, threshold=100):
    def transform(p):
        c = p * (100 - (d_high if p >= threshold else d_low))
        return f"${c // 100}.{c % 100:02d}"
    return " ".join(transform(int(w[1:])) if classify(w) else w for w in sentence.split(" "))

print(tiered("$50 $100 $250 bag$", 20, 10))

$45.00 $80.00 $200.00 bag$


### 15.4 Biến thể D — Đếm và tổng giá trị trước/sau giảm (không sửa câu)

In [32]:
def price_stats(sentence, discount):
    ps = [int(w[1:]) for w in sentence.split(" ") if classify(w)]
    before = sum(ps) * 100; after = sum(p * (100 - discount) for p in ps)
    fmt = lambda c: f"{c // 100}.{c % 100:02d}"
    return len(ps), fmt(before), fmt(after)

print(price_stats(examples[0][0], 50), price_stats(examples[1][0], 100))

(2, '3.00', '1.50') (4, '23.00', '0.00')


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Ưu / nhược |
|---|---|---|---|---|
| 1. Tham chiếu `Fraction` | số hữu tỉ chính xác | O(L) | O(L) | chậm, dùng để kiểm chứng |
| 2. Tách + float | doocs | O(L) | O(L) | đúng nhờ giới hạn 10 chữ số |
| 3. Tách + số nguyên | xu = p·(100−d) | O(L) | O(L) | **mặc định**: chính xác tuyệt đối |
| 4. `re.sub` + lookaround | không tách | O(L) | O(L) | gọn, dễ sai ranh giới |
| 5. Quét tay | máy trạng thái | O(L) | O(L) | dùng ở ngôn ngữ khác / stream |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `discountPrices("$5 $$5 5$ $5$", 20)` và `discountPrices("$123", 3)`?

**Bài 2 (tính tay):** Tính `discountPrices("$999 x $1000", 15)` bằng số nguyên xu.

**Bài 3 (điền TODO):** Viết `bt3_todo(w)` trả về `True` ⇔ `w` là giá theo định nghĩa của đề, **không** dùng `isdigit` hay regex.

**Bài 4 (tự viết lại):** Viết `bt4_todo(sentence, discount)` cài bản nộp không nhìn code (số nguyên xu).

**Bài 5 (mini challenge):** Viết `bt5_todo(sentence, discount)` cho câu **không chuẩn** (nhiều dấu cách liên tiếp, có dấu cách đầu/cuối) mà vẫn **giữ nguyên mọi dấu cách**, chỉ thay các giá.

In [33]:
def bt3_todo(w):
    # TODO: w có phải giá không
    pass

def bt4_todo(sentence, discount):
    # TODO: bản nộp
    pass

def bt5_todo(sentence, discount):
    # TODO: giữ nguyên khoảng trắng
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: chỉ `"$5"` và `"$123"` là giá. Mức 2: `5 · 80 = 400` xu; `123 · 97 = 11931` xu. Mức 3: `"$4.00 $$5 5$ $5$"`, `"$119.31"`.

**Bài 2.** Mức 1: `keep = 85`. Mức 2: `999 · 85 = 84915`, `1000 · 85 = 85000`. Mức 3: `"$849.15 x $850.00"`.

**Bài 3.** Mức 1: độ dài ≥ 2. Mức 2: ký tự đầu là `$`. Mức 3: mọi ký tự còn lại thuộc `'0'..'9'`.

**Bài 4.** Mức 1: `split(" ")`. Mức 2: `c = int(w[1:]) * (100 - d)`. Mức 3: `f"${c // 100}.{c % 100:02d}"`, `" ".join`.

**Bài 5.** Mức 1: `split(" ")` **giữ** các chuỗi rỗng giữa các dấu cách liên tiếp. Mức 2: chuỗi rỗng không phải giá ⇒ giữ nguyên. Mức 3: `" ".join` dựng lại đúng mọi dấu cách — thực ra bản nộp dùng `split(" ")` đã đúng sẵn; hoặc dùng `re.sub` với lookaround.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `"$4.00 $$5 5$ $5$"`, `"$119.31"`. **Bài 2:** `"$849.15 x $850.00"`.

In [34]:
assert Solution().discountPrices("$5 $$5 5$ $5$", 20) == "$4.00 $$5 5$ $5$"
assert Solution().discountPrices("$123", 3) == "$119.31"
assert Solution().discountPrices("$999 x $1000", 15) == "$849.15 x $850.00"

def bt3(w):
    if len(w) < 2 or w[0] != "$": return False
    for c in w[1:]:
        if not ("0" <= c <= "9"): return False
    return True

def bt4(sentence, discount):
    ws = sentence.split(" ")
    for i, w in enumerate(ws):
        if bt3(w):
            c = int(w[1:]) * (100 - discount); ws[i] = f"${c // 100}.{c % 100:02d}"
    return " ".join(ws)

def bt5(sentence, discount):
    return bt4(sentence, discount)                      # split(" ") giữ chuỗi rỗng => dấu cách được bảo toàn

rng = random.Random(1986)
for _ in range(2000):
    ws = [("$" + str(rng.randint(1, 10**5))) if rng.random() < 0.4 else "".join(rng.choice("$9a") for _ in range(rng.randint(0, 4)))
          for _ in range(rng.randint(1, 6))]
    d = rng.randint(0, 100)
    for w in ws: assert bt3(w) == bool(re.fullmatch(r"\$[0-9]+", w))
    s_norm = " ".join(w for w in ws if w)
    if s_norm: assert bt4(s_norm, d) == reference(s_norm, d)
    s_dirty = " ".join(ws)                              # có thể có nhiều dấu cách liên tiếp / đầu / cuối
    assert bt5(s_dirty, d) == regex_sub(s_dirty, d)
print(repr(bt5("  $10   a$ $3 ", 50)))
print("Đáp án bài tập: tất cả assert OK")

'  $5.00   a$ $1.50 '
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
keep = 100 - discount
ws = sentence.split(" ")
for i, w in enumerate(ws):
    if len(w) > 1 and w[0] == "$" and w[1:].isdigit():   # khớp TOÀN BỘ từ
        c = int(w[1:]) * keep                            # giá mới tính bằng xu (số nguyên)
        ws[i] = f"${c // 100}.{c % 100:02d}"             # luôn đúng 2 chữ số thập phân
return " ".join(ws)
```

**Checklist:**
- [ ] Nhân với `100 − discount` (giá còn lại), không phải `discount`.
- [ ] Từ `"$"` đứng một mình không phải giá; `"$10$"`, `"$$1"`, `"5$"` cũng không.
- [ ] Đúng 2 chữ số thập phân (`:02d` / `.2f`), không dùng `round` + `str`.
- [ ] Không `str.replace` trên cả câu; regex thì không dùng `\b` quanh `$`.